In [1]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO
import math

url = 'https://files.luxms.com/s/zyybXGNi82co9lc/download'  # добавь /download для Nextcloud

response = requests.get(url)
response.raise_for_status()  # проверка на ошибки HTTP

# Читаем Excel из памяти, не сохраняя на диск
df = pd.read_excel(BytesIO(response.content), engine='openpyxl')
df

,product,part_ready,name_full,description,lic_price,support_price,fstek,tl,lic_type,support_descr,support_name,support_part,PD
0,Luxms BI,LXBI-DCE-VUL,Luxms BI DataCenter Edition Viewer User License,Пользователь с базовым доступом на просмотр че...,48000.0,14400.0,Нет,Нет,DCE,Техническая поддержка 1 год на лицензию Luxms ...,Luxms BI DataCenter Edition Viewer User Licens...,LXBI-DCE-VUL-S1Y,PROD
1,Luxms BI,LXBI-DCE-CUL,Luxms BI DataCenter Edition Creator User License,Пользователь с расширенным доступом: ВУК Luxms...,80000.0,24000.0,Нет,Нет,DCE,Техническая поддержка 1 год на лицензию Luxms ...,Luxms BI DataCenter Edition Creator User Licen...,LXBI-DCE-CUL-S1Y,PROD
2,Luxms BI,LXBI-DCE-AUL,Luxms BI DataCenter Edition Admin User License,Пользователь с доступом к администрированию си...,84000.0,25200.0,Нет,Нет,DCE,Техническая поддержка 1 год на лицензию Luxms ...,Luxms BI DataCenter Edition Admin User License...,LXBI-DCE-AUL-S1Y,PROD
3,Luxms BI,LXBI-DCE-DEV-VUL,Luxms BI DataCenter Edition DEV Viewer User Li...,Пользователь с базовым доступом на просмотр че...,24000.0,7200.0,Нет,Нет,DCE,Техническая поддержка 1 год на лицензию Luxms ...,Luxms BI DataCenter Edition DEV Viewer User Li...,LXBI-DCE-DEV-VUL-S1Y,DEV
4,Luxms BI,LXBI-DCE-DEV-CUL,Luxms BI DataCenter Edition DEV Creator User L...,Пользователь с расширенным доступом для непрод...,40000.0,12000.0,Нет,Нет,DCE,Техническая поддержка 1 год на лицензию Luxms ...,Luxms BI DataCenter Edition DEV Creator User L...,LXBI-DCE-DEV-CUL-S1Y,DEV
...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,DCBI Luxms,DCBI-L-D-PL-U-D,Право на использование ПО Datacatalog BI Luxms...,Бессрочная лицензия ПО Datacatalog BI Luxms Ed...,33000.0,9900.0,Нет,Нет,DCE,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-D-PL-U-D-1Y-8,DEV
69,DCBI Luxms,DCBI-L-P-PL-EСUL,Право на использование ПО Datacatalog BI Luxms...,Лицензия сроком на 1 год ПО Datacatalog BI Lux...,22032000.0,6609600.0,Нет,Нет,EC,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-P-PL-EСUL-1Y-8,PROD
70,DCBI Luxms,DCBI-L-P-TL-U-R-1Y,Право на использование ПО Datacatalog BI Luxms...,Временная лицензия ПО Datacatalog BI Luxms Edi...,14400.0,4320.0,Нет,Да,DCE,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-P-TL-U-R-1Y-8,PROD
71,DCBI Luxms,DCBI-L-P-TL-U-A-1Y,Право на использование ПО Datacatalog BI Luxms...,Временная лицензия ПО Datacatalog BI Luxms Edi...,30000.0,9000.0,Нет,Да,DCE,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-P-TL-U-A-1Y-8,PROD


In [2]:
prod_admins= 10
prod_viewers= 50
prod_creators= 1000
prod_servers= 1

dev_admins= 2
dev_viewers= 5
dev_creators= 500
dev_servers= 1

In [3]:

df2=df.copy()

pd.set_option('display.max_rows', len(df2))
pd.set_option('display.max_columns', 14) 
pd.set_option('display.float_format', '{:,.2f}'.format)


# Базовые условия
base_cond = (df2['lic_type'] == 'DCE')

# Проверяем наличие "DEV" в любом месте строки
is_dev = (df2['PD'] == 'DEV')

# Маска подписки
is_tl = (df2['tl'] == 'Да')

# Условия для разных типов лицензий и контуров
aul_prod = base_cond & df2['part_ready'].str.contains('AUL', case=False, na=False) & ~is_dev & ~is_tl
dcbi_aul_prod = base_cond & df2['part_ready'].str.contains('PL-U-A', case=False, na=False) & ~is_dev & ~is_tl
aul_dev = base_cond & df2['part_ready'].str.contains('AUL', case=False, na=False) & is_dev & ~is_tl
dcbi_aul_dev = base_cond & df2['part_ready'].str.contains('PL-U-D', case=False, na=False) & is_dev & ~is_tl

vul_prod = base_cond & df2['part_ready'].str.contains('VUL', case=False, na=False) & ~is_dev & ~is_tl
vul_dev = base_cond & df2['part_ready'].str.contains('VUL', case=False, na=False) & is_dev & ~is_tl
dcbi_vul_dev = base_cond & df2['part_ready'].str.contains('PL-U-R', case=False, na=False) & ~is_dev & ~is_tl

cul_prod = base_cond & df2['part_ready'].str.contains('CUL', case=False, na=False) & ~is_dev & ~is_tl
cul_dev = base_cond & df2['part_ready'].str.contains('CUL', case=False, na=False) & is_dev & ~is_tl

ee_prod = (df2['lic_type']=='EE') & ~is_dev & ~is_tl
ee_dev = (df2['lic_type']=='EE') & is_dev & ~is_tl

ec_prod = (df2['lic_type']=='EC') & ~is_dev & ~(df2['part_ready'] == 'DCBI-L-P-PL-UC10') & ~is_tl
ec_dev = (df2['lic_type']=='EC') & is_dev & ~is_tl

ee_l350 = (df2['lic_type']=='EE-L350') & ~is_dev & ~is_tl
ec_l350 = (df2['lic_type']=='EC-L350') & ~is_dev & ~is_tl

# 10 серверов DCBI
uc10_cond = (df2['part_ready'] == 'DCBI-L-P-PL-UC10') & ~is_tl

# Подписка DCE
tl_aul_prod = base_cond & df2['part_ready'].str.contains('AUL', case=False, na=False) & ~is_dev & is_tl
dcbi_tl_aul_prod = base_cond & df2['part_ready'].str.contains('TL-U-A', case=False, na=False) & ~is_dev & is_tl
tl_aul_dev = base_cond & df2['part_ready'].str.contains('AUL', case=False, na=False) & is_dev & is_tl
dcbi_tl_aul_dev = base_cond & df2['part_ready'].str.contains('TL-U-D', case=False, na=False) & is_dev & is_tl

tl_vul_prod = base_cond & df2['part_ready'].str.contains('VUL', case=False, na=False) & ~is_dev & is_tl
tl_vul_dev = base_cond & df2['part_ready'].str.contains('VUL', case=False, na=False) & is_dev & is_tl
dcbi_tl_vul_dev = base_cond & df2['part_ready'].str.contains('TL-U-R', case=False, na=False) & ~is_dev & is_tl

tl_cul_prod = base_cond & df2['part_ready'].str.contains('CUL', case=False, na=False) & ~is_dev & is_tl
tl_cul_dev = base_cond & df2['part_ready'].str.contains('CUL', case=False, na=False) & is_dev & is_tl

tl_ee_prod = (df2['lic_type']=='EE') & ~is_dev & is_tl
tl_ee_dev = (df2['lic_type']=='EE') & is_dev & is_tl

tl_ec_prod = (df2['lic_type']=='EC') & ~is_dev & is_tl
tl_ec_dev = (df2['lic_type']=='EC') & is_dev & is_tl

conditions = [
    aul_prod, dcbi_aul_prod, aul_dev, dcbi_aul_dev, vul_prod, vul_dev, dcbi_vul_dev, cul_prod, cul_dev, ee_prod, 
    ee_dev, ec_prod, ec_dev, ee_l350, ec_l350, uc10_cond, tl_aul_prod, dcbi_tl_aul_prod, tl_aul_dev, dcbi_tl_aul_dev,
    tl_vul_prod, tl_vul_dev, dcbi_tl_vul_dev, tl_cul_prod, tl_cul_dev,
    tl_ee_prod, tl_ee_dev, tl_ec_prod, tl_ec_dev
              ]

choices = [
    df2['lic_price'] * prod_admins,   # AUL PROD
    df2['lic_price'] * prod_creators,   # DCBI AUL PROD
    df2['lic_price'] * dev_admins,   # AUL DEV
    df2['lic_price'] * (dev_admins+dev_viewers+dev_creators),   # DCBI AUL DEV

    df2['lic_price'] * prod_viewers,  # VUL PROD
    df2['lic_price'] * dev_viewers,  # VUL DEV
    df2['lic_price'] * prod_viewers,  # DCBI VUL PROD

    df2['lic_price'] * prod_creators,  # CUL PROD
    df2['lic_price'] * dev_creators,  # CUL DEV

    df2['lic_price'] * max(math.ceil((prod_admins+prod_viewers+prod_creators)/500), prod_servers), # EE PROD
    df2['lic_price'] * max(math.ceil((dev_admins+dev_viewers+dev_creators)/500), dev_servers, 1, math.ceil((prod_admins+prod_viewers+prod_creators)/500+1)), # EE DEV

    df2['lic_price'] * max(math.ceil((prod_admins+prod_viewers+prod_creators)/1000), prod_servers), # EC PROD
    df2['lic_price'] * max(math.ceil((dev_admins+dev_viewers+dev_creators)/500), dev_servers, 2, math.ceil((prod_admins+prod_viewers+prod_creators)/1000+1)), # EC DEV

    df2['lic_price'] * max(math.ceil((prod_admins+prod_viewers+prod_creators)/350), prod_servers), # EE_L350 PROD
    df2['lic_price'] * max(math.ceil((prod_admins+prod_viewers+prod_creators)/350), prod_servers), # EC_L350 PROD

    df2['lic_price'] * prod_servers, # 10 серверов DCBI

    df2['lic_price'] * prod_admins, # TL AUL PROD
    df2['lic_price'] * prod_creators, # TL DCBI AUL PROD
    df2['lic_price'] * dev_admins, # TL AUL DEV
    df2['lic_price'] * (dev_admins+dev_viewers+dev_creators), # TL DCBI AUL DEV

    df2['lic_price'] * prod_viewers,  # TL VUL PROD
    df2['lic_price'] * dev_viewers,  # TL VUL DEV
    df2['lic_price'] * prod_viewers,  # TL DCBI VUL PROD

    df2['lic_price'] * prod_creators,  # TL CUL PROD
    df2['lic_price'] * dev_creators,  # TL CUL DEV

    df2['lic_price'] * max(math.ceil((prod_admins+prod_viewers+prod_creators)/500), prod_servers), # TL EE PROD
    df2['lic_price'] * max(math.ceil((dev_admins+dev_viewers+dev_creators)/500), dev_servers, 1, math.ceil((prod_admins+prod_viewers+prod_creators)/500+1)), # TL EE DEV

    df2['lic_price'] * max(math.ceil((prod_admins+prod_viewers+prod_creators)/1000), prod_servers), # TL EC PROD
    df2['lic_price'] * max(math.ceil((dev_admins+dev_viewers+dev_creators)/500), dev_servers, 2, math.ceil((prod_admins+prod_viewers+prod_creators)/1000+1)), # TL EC DEV
]

df2['summ'] = np.select(conditions, choices, default=np.nan)

print(len(df2))
df2[df2['product']=='DCBI Luxms']


73


,product,part_ready,name_full,description,lic_price,support_price,fstek,tl,lic_type,support_descr,support_name,support_part,PD,summ
61,DCBI Luxms,DCBI-L-P-PL-U-R,Право на использование ПО Datacatalog BI Luxms...,Бессрочная лицензия ПО Datacatalog BI Luxms Ed...,"48,000.00","14,400.00",Нет,Нет,DCE,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-P-PL-U-R-1Y-8,PROD,"2,400,000.00"
62,DCBI Luxms,DCBI-L-P-PL-U-A,Право на использование ПО Datacatalog BI Luxms...,Бессрочная лицензия ПО Datacatalog BI Luxms Ed...,"84,000.00","25,200.00",Нет,Нет,DCE,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-P-PL-U-A-1Y-8,PROD,"84,000,000.00"
63,DCBI Luxms,DCBI-L-P-PL-EEUL,Право на использование ПО Datacatalog BI Luxms...,Бессрочная лицензия Datacatalog BI Luxms editi...,"12,960,000.00","3,888,000.00",Нет,Нет,EE,"Сопровождение ПО Datacatalog BI Luxms edition,...","Сопровождение ПО Datacatalog BI Luxms edition,...",DCBI-S-P-PL-EEUL-1Y-8,PROD,"38,880,000.00"
64,DCBI Luxms,DCBI-L-P-PL-EEL350,Право на использование ПО Datacatalog BI Luxms...,Бессрочная лицензия Datacatalog BI Luxms editi...,"6,480,000.00","1,944,000.00",Нет,Нет,EE-L350,"Сопровождение ПО Datacatalog BI Luxms edition,...","Сопровождение ПО Datacatalog BI Luxms edition,...",DCBI-S-P-PL-EEL350-1Y-8,PROD,"25,920,000.00"
65,DCBI Luxms,DCBI-L-D-PL-DEVL,Право на использование ПО Datacatalog BI Luxms...,Бессрочная лицензия Datacatalog BI Luxms editi...,"6,480,000.00","1,944,000.00",Нет,Нет,EC,"Сопровождение ПО Datacatalog BI Luxms edition,...","Сопровождение ПО Datacatalog BI Luxms edition,...",DCBI-S-D-PL-DEVL-1Y-8,DEV,"19,440,000.00"
66,DCBI Luxms,DCBI-L-D-PL-DEVL,Право на использование ПО Datacatalog BI Luxms...,Бессрочная лицензия Datacatalog BI Luxms editi...,"6,480,000.00","1,944,000.00",Нет,Нет,EE,"Сопровождение ПО Datacatalog BI Luxms edition,...","Сопровождение ПО Datacatalog BI Luxms edition,...",DCBI-S-D-PL-DEVL-1Y-8,DEV,"25,920,000.00"
67,DCBI Luxms,DCBI-L-P-PL-UC10,Неисключительное право на использование ПО Dat...,Бессрочная лицензия для построения решений. Вк...,"65,000,000.00","19,500,000.00",Нет,Нет,EC,Сопровождение ПО,Сопровождение ПО для лицензии DCBI-L-P-PL-UC1...,DCBI-S-P-PL-UC10-1Y-8,PROD,"65,000,000.00"
68,DCBI Luxms,DCBI-L-D-PL-U-D,Право на использование ПО Datacatalog BI Luxms...,Бессрочная лицензия ПО Datacatalog BI Luxms Ed...,"33,000.00","9,900.00",Нет,Нет,DCE,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-D-PL-U-D-1Y-8,DEV,"16,731,000.00"
69,DCBI Luxms,DCBI-L-P-PL-EСUL,Право на использование ПО Datacatalog BI Luxms...,Лицензия сроком на 1 год ПО Datacatalog BI Lux...,"22,032,000.00","6,609,600.00",Нет,Нет,EC,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-P-PL-EСUL-1Y-8,PROD,"44,064,000.00"
70,DCBI Luxms,DCBI-L-P-TL-U-R-1Y,Право на использование ПО Datacatalog BI Luxms...,Временная лицензия ПО Datacatalog BI Luxms Edi...,"14,400.00","4,320.00",Нет,Да,DCE,Сопровождение ПО Datacatalog BI Luxms Edition ...,Сопровождение ПО Datacatalog BI Luxms Edition ...,DCBI-S-P-TL-U-R-1Y-8,PROD,"720,000.00"
